# 트레이너가 두 번 짚은 골반 위치와 예비 판독 비교
학습 영상 b04의 이륙 후보 9번과 무릎 통과 21번, a01의 이륙 후보 5번을 비교합니다. [수동 판독 노트북](../annotation/05_train_hip_review.ipynb)에서 두 회차를 저장한 뒤 실행하세요. 소켓은 화면 단서로 추정한 위치입니다.

### 1. 같은 프레임의 네 점 읽기
원본의 왼쪽 위가 (0, 0)입니다. 좌표는 오른쪽과 아래로 갈수록 커집니다. 판독 불가로 저장한 점은 좌표가 비어 있습니다.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from wlcv.annotation.pose_points import load_points, pose_point_path
from wlcv.config import ANNOTATIONS, ROOT, VIDEOS
from wlcv.evaluation.hip_visibility import plot_hip_reads

frames = [("b04_sn_f_01", 9), ("b04_sn_f_01", 21), ("a01_sn_f_01", 5)]
audit = pd.read_csv(ROOT / "data/metadata/train_hip_visibility.csv").set_index(["video_id", "frame_index"])
files = pd.read_csv(ROOT / "data/metadata/videos.csv").set_index("video_id").filename
folder = ANNOTATIONS / "train_hip_review"
output = ROOT / "outputs/pose_baseline/train_hip_repeat"
output.mkdir(parents=True, exist_ok=True)
records = []
for video, frame in frames:
    event = audit.loc[(video, frame)]
    for number in (1, 2):
        path = pose_point_path(folder, video, frame, pass_number=number)
        manual = load_points(path).get("골반")
        if manual is None:
            raise ValueError(f"{video} {frame}번 {number}차 수동 판독을 먼저 저장하세요.")
        assert (manual["video_id"], int(manual["frame_index"]), manual["side"], manual["split"]) == (
            video, frame, event.near_side, "train"
        )
        records.append((video, frame, "트레이너", number, manual["status"], manual["reason"],
                        manual["x_px"], manual["y_px"]))
        records.append((video, frame, "예비", number, event.screen, "",
                        event[f"read{number}_x"], event[f"read{number}_y"]))
points = pd.DataFrame(records, columns=["video_id", "frame_index", "reader", "round_number",
                                       "status", "reason", "x", "y"])
points[["x", "y"]] = points[["x", "y"]].replace("", np.nan).astype(float)
display(points)
points.to_csv(output / "points.csv", index=False)

### 2. 반복 차이와 판독자 사이 차이
반복 거리는 한 판독자의 1·2차 점 사이 직선거리입니다. 판독자끼리는 각각의 두 점을 평균한 중점으로 비교합니다. 예비 판독과 수동 판독은 독립적으로 진행했으므로 회차 번호끼리 대응시키지 않습니다. 한 점이라도 비면 해당 비교는 계산하지 않습니다.

In [ ]:
summary = []
for (video, frame), group in points.groupby(["video_id", "frame_index"], sort=False):
    human = group.loc[group.reader == "트레이너", ["x", "y"]].to_numpy()
    prelim = group.loc[group.reader == "예비", ["x", "y"]].to_numpy()
    dx, dy = human.mean(axis=0) - prelim.mean(axis=0)
    summary.append((video, frame, np.linalg.norm(human[1] - human[0]),
                    np.linalg.norm(prelim[1] - prelim[0]), dx, dy, np.hypot(dx, dy)))
comparison = pd.DataFrame(summary, columns=[
    "영상", "프레임", "트레이너 반복 거리", "예비 반복 거리", "중점 차이 x", "중점 차이 y", "중점 거리"
])
display(comparison.round(1))
comparison.to_csv(output / "comparison.csv", index=False)

### 3. 원본에 네 점 겹쳐 보기
노랑은 트레이너, 하늘색은 예비 판독입니다. 원은 1차, 십자는 2차입니다. 아래 행은 각 원본에서 150 × 150픽셀 범위를 확대했습니다. 세 장의 반복 거리로 해부학적 정답이나 공통 허용 오차를 정할 수는 없습니다.

In [ ]:
fig = plot_hip_reads(points, files, VIDEOS)
fig.savefig(output / "hip_reads.png", dpi=140, bbox_inches="tight")
plt.show()